# FTS ML Backtesting & Interactive Model Inspector

Welcome to the interactive backtesting workspace! This notebook shows you how to:
1. **Load historical data** into a local SQLite database.
2. **Train a machine learning model** (RNN) from the `nets` plugin.
3. **Run a backtest simulation** with a SOLID `DatabasePredictionLogger` callback registered on the strategy to log prediction outputs.
4. **Render the interactive dashboard** using Plotly and ipywidgets to inspect model details, weights, P&L, and sizing.

### 1. Import Dependencies

In [1]:
import os
from sqlalchemy import create_engine
from datetime import datetime, timezone

from trading_bot.core.database import init_db, SessionLocal
from trading_bot.backtesting import BacktestSimulator, BacktestVisualizer
from trading_bot.core.repository import MarketDataRepository                   
from trading_bot.core.schemas import BarData                     

from nets.output_selectors import DynamicThresholdClassifier
from nets.inference import ONNXPredictor
from nets.strategies.nets_strategy import NetsStrategy
from nets.models import LSTMConfig, NNTrainingConfig
from nets.training import LSTMTrainer
from trading_bot.core.transforms import LogReturnTransform
from trading_bot.strategy.engine import StrategyEngine


### 2. Setup SQLite Database & Load Historical Data

We initialize the schema (including our new `ModelPredictionLog` table) and copy the historical BTC/USDT data from `tests/test_persistence.db` into our active `dev.db` database.

In [6]:
db_url = 'sqlite:///../dev.db'                                                                                      
                                                                                                            
# Initialize schema and create tables                                                                     
init_db()                                                                                                 
                                                                                                            
# OPTIONAL: You can clear order, trade, and position logs                                                 
# while PRESERVING the historical bar_data_logs we just ingested.                                         
dest_engine = create_engine(db_url)                                                                         
print("Database prepared (preserving historical bars).")

dest_engine = create_engine(db_url, pool_pre_ping=True)
SessionLocal.configure(bind=dest_engine)

db = SessionLocal()


Database prepared (preserving historical bars).


In [7]:
repo = MarketDataRepository(db)                                                                           
# Fetch specifically the 30-minute bars we downloaded                                                     
db_bars = repo.get_bars('BTC/USDT', interval='30m')                                                       

bar_schemas = [                                                                                           
    BarData(                                                                                              
        timestamp=bar.timestamp,                                                                          
        open=bar.open,                                                                                    
        high=bar.high,                                                                                    
        low=bar.low,                                                                                      
        close=bar.close,                                                                                  
        volume=bar.volume,                                                                                
        bar_type=bar.bar_type,                                                                            
        ticks_count=bar.ticks_count,                                                                      
        dollar_volume=bar.dollar_volume,                                                                  
    )                                                                                                     
    for bar in db_bars                                                                                    
]                                                                                                         
print(f"Loaded {len(bar_schemas)} bars for training.")        


Loaded 1000 bars for training.


### 3. Train a Machine Learning Model

We train a recurrent neural network (RNN) using the `nets` plugin's `RNNTrainer` on the historical bars, and save it as an ONNX model.

In [8]:
print('Training model...')
model_config = LSTMConfig(                                                                                
    hidden_dim=64,       # Increased capacity (represents state memory dimension)                         
    num_layers=2,        # Stacked layers to capture hierarchical patterns                                
    dropout=0.1,         # Prevents overfitting by randomly shutting off 10% of node connections          
    bidirectional=False  # Must be False for causal time series (cannot see the future!)                  
)          
training_config = NNTrainingConfig(
    epochs=50,
    batch_size=32,
    learning_rate=0.01,
    validation_split=0.1,                                       
    tensorboard_log_dir="runs/my_lstm_run"
)

trainer = LSTMTrainer(
    lookback_period=20,
    model_config=model_config,
    training_config=training_config,
)

onnx_bytes = trainer.train(bar_schemas)

os.makedirs('models', exist_ok=True)
onnx_path = 'models/my_lstm_model.onnx'
with open(onnx_path, 'wb') as f:
    f.write(onnx_bytes)

print(f'Model trained and exported to: {onnx_path}')

Training model...


/home/alfred/github/fts/src/plugins/nets/training/training.py:558: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0621 21:25:23.589000 41687 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0621 21:25:23.591000 41687 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0621 21:25:23.591000 41687 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `SimpleLSTM([...]` with `torch.export.export(..., strict=False)`...


/home/alfred/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/contextlib.py:144: UserWarning: The tensor attributes self.lstm._flat_weights[0], self.lstm._flat_weights[1], self.lstm._flat_weights[2], self.lstm._flat_weights[3], self.lstm._flat_weights[4], self.lstm._flat_weights[5], self.lstm._flat_weights[6], self.lstm._flat_weights[7] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `SimpleLSTM([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model trained and exported to: models/my_lstm_model.onnx


/home/alfred/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


### 4. Run Backtest Simulation with Callback Logging

We instantiate the `BacktestSimulator` using `NetsStrategy` loaded with the trained ONNX model. We register a `DatabasePredictionLogger` as an observer on the strategy. As the simulation replays the historical ticks, the strategy notifies the logger, which saves raw predictions to the database—without modifying core bot or simulator loops.

In [9]:
# 1. Initialize predictor, transform, and output selector
predictor = ONNXPredictor(onnx_path)
transform = LogReturnTransform()

# Threshold = k * ATR percentage over the last 10 periods
output_selector = DynamicThresholdClassifier(k=0.03, period=10, confidence_multiplier=20.0)

# 2. Define Nets strategy
strategy = NetsStrategy(
    predictor=predictor,
    transform=transform,
    output_selector=output_selector,
    lookback_period=20,
    name_suffix='lstm',
    feature_cols=['close']
)

# 3. Setup StrategyEngine
strategy_engine = StrategyEngine(strategies=[strategy])

# 4. Define backtest dates based on the ingested data
start_date = datetime(2025, 10, 30, 20, 0, 0, tzinfo=timezone.utc)
end_date = datetime(2026, 6, 19, 5, 30, 0, tzinfo=timezone.utc)

# 5. Initialize the simulator
simulator = BacktestSimulator(
    db=db,
    strategy_engine=strategy_engine,
    market_ids=['BTC/USDT'],
    start_date=start_date,
    end_date=end_date
)

# 6. Run the simulation
print('Running backtest simulation...')
simulator.run()

db.close()

Running backtest simulation...


### 5. Render Interactive Dashboard

We load our interactive `BacktestVisualizer` and point it to the SQLite database file. We also pass our trained ONNX model path to enable layer weight visualization.

In [10]:
# Instantiate the visualizer
viz = BacktestVisualizer('sqlite:///../dev.db')

# Display the dashboard
viz.show_dashboard(onnx_model_path=onnx_path)

Output()

Output()

HTML(value="<hr style='border-color:#37474f;'/>")

HTML(value='<h3>🧬 ONNX Model Weight Inspector</h3>')

Output()